# VERIMETER: Institutional Verification Diagnostics Demo

This notebook walks through the key concepts and statistical diagnostics implemented in `verimeter`. We cover:
1. Setting up the packages and paths.
2. Simulating a capacity inversion (the sub-proportional staffing problem).
3. Simulating spurious regressions on independent random walks (and how the Engle-Granger gate solves it).
4. Capture-recapture two-screen modeling and screen dependence limits.
5. Running the empirical EOIR Workload pipeline.

## 1. Imports and Setup

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add src directory to path
sys.path.insert(0, os.path.abspath('../src'))
sys.path.insert(0, os.path.abspath('..'))

import verimeter as V
from simulation.data_generator import generate_panel_data, generate_capture_recapture_data
print("VERIMETER version:", V.__version__)

## 2. The Capacity Inversion (Sub-Proportional Staffing)

We simulate an institution with sub-proportional staffing (elasticity $\beta = 0.30$) where the true error rate is constant ($q = 8\%$), but caseload grows tenfold. We show how the reported rate drops dramatically, creating an illusion of improvement.

In [ ]:
# Generate panel data
panel = generate_panel_data(n_periods=30, beta=0.30, q_true=0.08, delta_true=0.60, seed=42)

# Run diagnostics
rep = V.diagnose(panel["caseload"], panel["examined"], panel["detected"])
print(rep)

## 3. Spurious Regression on Independent Random Walks

Traditional regression models find a strong relationship between independent trending series (spurious regression). We show that standard OLS declares a capacity inversion for independent series, whereas our Engle-Granger gate flags it as spurious.

In [ ]:
np.random.seed(123)
n_periods = 40
# Generate independent random walks
log_lam = np.cumsum(np.random.normal(0.05, 0.1, n_periods))
log_kap = np.cumsum(np.random.normal(0.05, 0.1, n_periods))
lam = np.exp(log_lam) * 1000.0
kap = np.exp(log_kap) * 500.0

# Run verimeter diagnostics on these unrelated series
try:
    ela = V.capacity_elasticity(lam, kap, require_cointegration=True)
    print("Estimated Beta:", ela.beta)
    print("Is Cointegrated:", ela.cointegrated)
    print("Is Reliable:", ela.reliable)
    print("Verdict:", ela.verdict)
except Exception as e:
    print("Error:", e)

## 4. Capture-Recapture Depth Modeling and Screen Dependence

A second independent screen identifies true error rates ($q$) and reviewer depth ($\delta$). We show how reviewer correlation affects our estimations.

In [ ]:
# Draw capture-recapture values with zero dependency
data_indep = generate_capture_recapture_data(n_overlap=2500, q_true=0.10, delta1=0.6, delta2=0.4, rho=0.0, seed=42)
est_indep = V.two_screen(data_indep["n11"], data_indep["n10"], data_indep["n01"], n_overlap=2500)
print("=== Independent Screens ===")
print(est_indep.summary())

# Draw capture-recapture values with high dependency (rho=0.6)
data_dep = generate_capture_recapture_data(n_overlap=2500, q_true=0.10, delta1=0.6, delta2=0.4, rho=0.6, seed=42)
est_dep = V.two_screen(data_dep["n11"], data_dep["n10"], data_dep["n01"], n_overlap=2500)
print("\n=== Dependent Screens (rho=0.6) ===")
print(est_dep.summary())

## 5. Running the Empirical EOIR Pipeline

We execute the workload pipeline, validating and diagnosing the processed panel.

In [ ]:
from empirical.pipeline import run_pipeline

panel_df = run_pipeline()
print("\nParsed EOIR Panel Header:")
print(panel_df.head())

rep_eoir = V.diagnose(panel_df["caseload"], panel_df["examined"], panel_df["detected"])
print("\n=== EOIR Diagnosis ===")
print(rep_eoir)